# Model 3: Generative LLM (advanced/exploratory)

Manually-labeled ABSA data

In this model, full text reviews are fed to a generative LLM which “reads” the reviews and returns aspect-sentiment pairs in a defined JSON format. The OpenAI model `gpt-4.1-mini` was selected because it offers strong performance on structured extraction tasks, reliable instruction following, and significantly lower cost and latency than larger models, making it practical for processing thousands of reviews via the API.

In this notebook, the models are tested on the manually-labeled "gold standard" ABSA data. See `07a` for this model's performance on the data labeled for weak supervision.

## OpenAI API Key

This notebook requires an OpenAI API key.

When running the notebook, you will be prompted to enter your key.

You can obtain a key from:
https://platform.openai.com/api-keys

Approximate cost and time of each API call is reported in the markdown text before the call.

In [1]:
# Install/upgrade the OpenAI Python client
%pip install -U openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Securely request OpenAI API Key from notebook user
import os
from getpass import getpass
from openai import OpenAI

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

# Select OpenAI model and prompt lag time
MODEL = "gpt-4.1-mini"
SLEEP_SECONDS = 0.2

Enter your OpenAI API key:  ········


## Imports

In [4]:
import json
import time
import random
import warnings
import pandas as pd
from collections import defaultdict

from tqdm import tqdm
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report
)

warnings.filterwarnings("ignore")

## B. Manual ABSA Evaluation

### B.1. Load and Prepare Manually Labelled ABSA Data

The manually labelled ABSA data is stored in two files: one containing sentence-level review text and metadata, and one containing long-format aspect-sentiment labels. Because labelling was performed at the sentence level, labels are first converted to binary aspect-sentiment columns and then aggregated to the review level so that repeated mentions of the same label within a review are counted only once.

In [ ]:
# ---------------------------------------------------------------------------------------
# MANUAL ABSA DATA FILES
# ---------------------------------------------------------------------------------------

MANUAL_TEXT_FILE = "../data/absa_training_set.csv"
MANUAL_LABELS_FILE = "../data/absa_labels_long.csv"

# Save paths
MANUAL_EVAL_OUTFILE = "../data/manual_absa_eval_reviews.csv"
MANUAL_ZERO_RESULTS_OUTFILE = "../data/openai_manual_zero_shot_results.csv"
MANUAL_FEW_RESULTS_OUTFILE = "../data/openai_manual_few_shot_results.csv"
MANUAL_COMPARISON_OUTFILE = "../data/openai_manual_comparison_metrics.csv"

In [ ]:
# ---------------------------------------------------------------------------------------
# HELPER MAPS FOR MANUAL LABEL CONVERSION
# ---------------------------------------------------------------------------------------

ASPECT_TO_PREFIX = {
    "Product Quality": "product_quality",
    "Service": "service",
    "Wait Time": "wait_time",
    "Price/Value": "price_value",
    "Cleanliness": "cleanliness",
    "Atmosphere": "atmosphere",
    "General": "general"
}

# Normalization for slightly different style in manual file
ASPECT_NORMALIZATION = {
    "food_quality": "Product Quality",
    "service": "Service",
    "wait_time": "Wait Time",
    "price_value": "Price/Value",
    "cleanliness": "Cleanliness",
    "atmosphere": "Atmosphere",
    "general": "General"
}

SENTIMENT_NORMALIZATION = {
    "positive": "positive",
    "negative": "negative"
}

In [ ]:
# ---------------------------------------------------------------------------------------
# FUNCTION TO BUILD REVIEW-LEVEL MANUAL ABSA EVALUATION SET
# ---------------------------------------------------------------------------------------

def build_manual_absa_review_eval_set(text_file, labels_file, label_cols):
    """
    Build a review-level manual ABSA evaluation set.

    Inputs:
      text_file:
        absa_training_set.csv with columns:
        review_id, gmap_id, rating, sentence_id, sentence_text

      labels_file:
        absa_labels_long.csv with columns:
        review_id, sentence_id, aspect, sentiment

    Output:
      one row per review_id with:
      - full review text reconstructed from ordered sentences
      - review metadata
      - 14 binary gold label columns matching LABEL_COLS

    Notes:
      - repeated labels within a review count once
      - positive/negative labels are aggregated with max()
      - if a review has both positive and negative for the same aspect,
        both binary columns are set to 1
    """
# -------------------------------------------------------------------
    # LOAD DATA
    # -------------------------------------------------------------------
    df_text = pd.read_csv(text_file)
    df_labels = pd.read_csv(labels_file)

    # Standardize ids as strings
    for col in ["review_id", "sentence_id"]:
        if col in df_text.columns:
            df_text[col] = df_text[col].astype(str)
        if col in df_labels.columns:
            df_labels[col] = df_labels[col].astype(str)

    if "gmap_id" in df_text.columns:
        df_text["gmap_id"] = df_text["gmap_id"].astype(str)

    # -------------------------------------------------------------------
    # CLEAN / NORMALIZE MANUAL LABELS
    # -------------------------------------------------------------------
    df_labels["aspect"] = (
        df_labels["aspect"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(ASPECT_NORMALIZATION)
    )


    df_labels["sentiment"] = (
        df_labels["sentiment"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(SENTIMENT_NORMALIZATION)
    )

    # Keep only valid rows
    df_labels = df_labels[
        df_labels["aspect"].notna() &
        df_labels["sentiment"].notna()
    ].copy()

    # -------------------------------------------------------------------
    # CONVERT LONG LABELS TO BINARY REVIEW-LEVEL UPDATES
    # -------------------------------------------------------------------

    label_updates = []

    for _, row in df_labels.iterrows():
        review_id = row["review_id"]
        aspect = row["aspect"]
        sentiment = row["sentiment"]

        if aspect not in ASPECT_TO_PREFIX:
            continue

        prefix = ASPECT_TO_PREFIX[aspect]

        if sentiment == "positive":
            label_updates.append((review_id, f"{prefix}_positive", 1))
        elif sentiment == "negative":
            label_updates.append((review_id, f"{prefix}_negative", 1))

    # -------------------------------------------------------------------
    # BUILD REVIEW-LEVEL GOLD LABEL FRAME
    # -------------------------------------------------------------------

    review_keys = df_text[["review_id"]].drop_duplicates().copy()

    for col in label_cols:
        review_keys[col] = 0

    if len(label_updates) > 0:
        df_updates = pd.DataFrame(
            label_updates,
            columns=["review_id", "label_col", "value"]
        ).drop_duplicates()

        df_updates_wide = (
            df_updates
            .pivot_table(
                index="review_id",
                columns="label_col",
                values="value",
                aggfunc="max",
                fill_value=0
            )
            .reset_index()
        )

        df_gold = review_keys.merge(df_updates_wide, on="review_id", how="left", suffixes=("", "__new"))



        for col in label_cols:
            new_col = f"{col}__new"
            if new_col in df_gold.columns:
                df_gold[col] = df_gold[[col, new_col]].max(axis=1)
                df_gold.drop(columns=[new_col], inplace=True)
    else:
        df_gold = review_keys.copy()

    # -------------------------------------------------------------------
    # RECONSTRUCT FULL REVIEW TEXT FROM SENTENCES
    # -------------------------------------------------------------------


    df_text = df_text.copy()

    # Try to sort by sentence_id in numeric order when possible
    df_text["_sentence_order"] = pd.to_numeric(df_text["sentence_id"], errors="coerce")
    df_text = df_text.sort_values(["review_id", "_sentence_order", "sentence_id"])

    df_reviews = (
        df_text.groupby("review_id", as_index=False)
        .agg({
            "gmap_id": "first",
            "rating": "first",
            "sentence_text": lambda x: " ".join(
                s.strip() for s in x.dropna().astype(str) if s.strip()
            )
        })
        .rename(columns={"sentence_text": "text"})
    )

    # -------------------------------------------------------------------
    # MERGE REVIEW TEXT + REVIEW-LEVEL GOLD LABELS
    # -------------------------------------------------------------------
    df_manual_eval = df_reviews.merge(df_gold, on="review_id", how="left")

    for col in label_cols:
        df_manual_eval[col] = df_manual_eval[col].fillna(0).astype(int)

    # -------------------------------------------------------------------
    # SUMMARY
    # -------------------------------------------------------------------
    print("Manual review-level ABSA evaluation set built.")
    print(f"Rows (reviews): {len(df_manual_eval):,}")
    print(f"Unique gmap_id: {df_manual_eval['gmap_id'].nunique():,}")

    print("\nPositive counts per binary label:")
    print(df_manual_eval[label_cols].sum().sort_values(ascending=False))

    return df_manual_eval


In [ ]:
# ------------------------------------------------------------
# BUILD MANUAL REVIEW-LEVEL EVALUATION SET
# ------------------------------------------------------------

df_manual_eval = build_manual_absa_review_eval_set(
    text_file=MANUAL_TEXT_FILE,
    labels_file=MANUAL_LABELS_FILE,
    label_cols=LABEL_COLS
)

df_manual_eval.to_csv(MANUAL_EVAL_OUTFILE, index=False)
print(f"\nSaved manual review-level evaluation set to: {MANUAL_EVAL_OUTFILE}")

display(df_manual_eval.head(3))

### B.2. Run Zero-Shot on the Manual ABSA Review Set

Zero-shot prompting is first applied to the manually labelled review set. Each reconstructed full review is sent to the model, and the returned aspect-sentiment pairs are mapped into the same 14 binary label columns used for evaluation.


In [ ]:
results_manual_zero = run_openai_absa(
    df_input=df_manual_eval,
    mode="zero_shot",
    sleep_seconds=SLEEP_SECONDS,
    save_every=100,
    outfile=MANUAL_ZERO_RESULTS_OUTFILE
)

### B.3. Run Few-Shot on the Manual ABSA Review Set

Few-shot prompting is then applied to the same manually labelled review set using the example review-response pairs defined earlier.

In [ ]:
results_manual_few = run_openai_absa(
    df_input=df_manual_eval,
    mode="few_shot",
    sleep_seconds=SLEEP_SECONDS,
    save_every=100,
    outfile=MANUAL_FEW_RESULTS_OUTFILE
)

### B.4. Evaluate Manual ABSA Results

The zero-shot and few-shot outputs are evaluated against the manually labelled review-level gold labels using subset accuracy, macro F1, weighted F1, and a full classification report.

In [ ]:
manual_zero_metrics = evaluate_multilabel_results(results_manual_zero, LABEL_COLS)
manual_few_metrics = evaluate_multilabel_results(results_manual_few, LABEL_COLS)

print_multilabel_results("MANUAL ABSA ZERO-SHOT METRICS", manual_zero_metrics)
print()
print_multilabel_results("MANUAL ABSA FEW-SHOT METRICS", manual_few_metrics)

### B.5. Compare Zero-Shot vs. Few-Shot on Manual ABSA Set

In [ ]:
# ------------------------------------------------------------
# FUNCTION TO CONVERT METRICS DICT TO A CLEAN COMPARISON ROW
# ------------------------------------------------------------

def metrics_to_row(mode_name, metrics):
    return {
        "mode": mode_name,
        "accuracy": metrics["accuracy"],
        "precision_weighted": metrics["precision"],
        "recall_weighted": metrics["recall"],
        "f1_macro": metrics["macro_f1"],
        "f1_weighted": metrics["weighted_f1"],
    }
manual_comparison = pd.DataFrame([
    metrics_to_row("zero_shot", manual_zero_metrics),
    metrics_to_row("few_shot", manual_few_metrics),
])

display(manual_comparison)

manual_comparison.to_csv(MANUAL_COMPARISON_OUTFILE, index=False)
print(f"\nSaved manual comparison metrics to: {MANUAL_COMPARISON_OUTFILE}")
B.6. Token Usage Summary for Manual ABSA Set
summarize_usage(results_manual_zero, "MANUAL ZERO-SHOT")
summarize_usage(results_manual_few, "MANUAL FEW-SHOT")

In [ ]:
# ------------------------------------------------------------
# REVIEW-LEVEL ERROR ANALYSIS
# Pull best / worst reviews based on row-level multilabel F1
# ------------------------------------------------------------

def add_row_level_scores(results_df, label_cols):
    """
    Add row-level precision, recall, F1, and exact-match flag.

    For each review:
    - precision = TP / (TP + FP)
    - recall    = TP / (TP + FN)
    - f1        = harmonic mean of precision and recall
    - exact_match = 1 if all labels match exactly, else 0
    """
    df = results_df.copy()

    true_cols = [f"true__{c}" for c in label_cols]
    pred_cols = [f"pred__{c}" for c in label_cols]

    row_precisions = []
    row_recalls = []
    row_f1s = []
    row_exact = []
    row_tp = []
    row_fp = []
    row_fn = []

    for _, row in df.iterrows():
        y_true = row[true_cols].astype(int).values
        y_pred = row[pred_cols].astype(int).values

        tp = ((y_true == 1) & (y_pred == 1)).sum()
        fp = ((y_true == 0) & (y_pred == 1)).sum()
        fn = ((y_true == 1) & (y_pred == 0)).sum()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        exact_match = int((y_true == y_pred).all())

        row_tp.append(tp)
        row_fp.append(fp)
        row_fn.append(fn)
        row_precisions.append(precision)
        row_recalls.append(recall)
        row_f1s.append(f1)
        row_exact.append(exact_match)

    df["row_tp"] = row_tp
    df["row_fp"] = row_fp
    df["row_fn"] = row_fn
    df["row_precision"] = row_precisions
    df["row_recall"] = row_recalls
    df["row_f1"] = row_f1s
    df["exact_match"] = row_exact

    return df


def get_true_pred_label_lists(row, label_cols):
    """
    Return lists of active true and predicted labels for one row.
    """
    true_labels = [col for col in label_cols if int(row[f"true__{col}"]) == 1]
    pred_labels = [col for col in label_cols if int(row[f"pred__{col}"]) == 1]
    return true_labels, pred_labels


def build_error_analysis_table(scored_df, label_cols, n=5, best=True):
    """
    Create a compact table of the best or worst rows.

    Ranking:
    - Best: highest row_f1, then highest exact_match, then lowest total mistakes
    - Worst: lowest row_f1, then lowest exact_match, then highest total mistakes
    """
    df = scored_df.copy()

    df["total_errors"] = df["row_fp"] + df["row_fn"]

    if best:
        df_ranked = df.sort_values(
            by=["row_f1", "exact_match", "total_errors"],
            ascending=[False, False, True]
        ).head(n)
    else:
        df_ranked = df.sort_values(
            by=["row_f1", "exact_match", "total_errors"],
            ascending=[True, True, False]
        ).head(n)

    rows = []
    for _, row in df_ranked.iterrows():
        true_labels, pred_labels = get_true_pred_label_lists(row, label_cols)

        rows.append({
            "review_id": row["review_id"],
            "rating": row.get("rating", None),
            "row_f1": round(row["row_f1"], 4),
            "row_precision": round(row["row_precision"], 4),
            "row_recall": round(row["row_recall"], 4),
            "exact_match": row["exact_match"],
            "tp": row["row_tp"],
            "fp": row["row_fp"],
            "fn": row["row_fn"],
            "text": row.get("text", ""),
            "true_labels": ", ".join(true_labels),
            "pred_labels": ", ".join(pred_labels),
            "raw_pairs": row.get("raw_pairs", "")
        })

    return pd.DataFrame(rows)